# 🧪 [Colab 실습] ONNX Export & 그래프 해부 — NPU 컴파일 직전의 관문

**온디바이스 AI 프로그래밍 · 「양자화 미니랩」 다음 단계 — Day 2 컴파일 파이프라인의 ②~③단계**

| 항목 | 내용 |
| --- | --- |
| 실습 목표 | PyTorch 모델을 ONNX로 내보내고, **교안의 함정 6가지를 일부러 밟아본 뒤**, 그래프를 해부·단순화·검증하는 실무 습관 획득 |
| 환경 | Google Colab — **CPU 런타임으로 충분** (학습 없음, 그래프 실습) |
| 핵심 도구 | `torch.onnx` · `onnx` · `onnxruntime` · `onnx-simplifier` · Netron |

## 이 실습의 위치 — Day 2 컴파일 6단계에서

```text
① 최적화 학습(QAT)  →  [② PyTorch Export  →  ③ ONNX 검증·간소화]  →  ④ 디퍼아이 컴파일러 → ...
   양자화 미니랩에서 완료      ★ 오늘 이 두 단계를 정복 ★                Day 2 본 실습
```

교안의 경고를 기억하세요:
> **"ONNX Export 함정 6가지 — PyTorch → ONNX 변환 시 NPU 컴파일 실패의 90%를 설명"**
> **"Opset ≥ 13, dynamic_axes 금지, 커스텀 연산자 교체 — 3가지가 NPU 변환 성공률을 결정"**

## 실습 로드맵

| Part | 주제 | 교안 대응 |
| --- | --- | --- |
| 1 | Export 대상 모델 준비 | — |
| 2 | 정석 Export 레시피 + 등가성 검증 습관 | 컴파일 ②단계 |
| 3 | ★ 함정 실험실 — 6가지를 일부러 밟아보기 | ONNX 함정 6가지 |
| 4 | onnx-simplifier 실전 | 컴파일 ③단계 |
| 5 | ★ Fallback 스캐너 직접 만들기 | Day 3 Fallback 대응 |
| 6 | Netron 시각화 | 컴파일 ③단계 |
| 7 | 리포트 & Day 2 연결 | — |


---
# Part 0. 환경 준비

### Step 0-1. 패키지 설치

Colab에 torch는 내장되어 있고, ONNX 3종 세트만 설치합니다 (약 1분).

In [ ]:
!pip install -q onnx onnxruntime onnxsim

import collections, copy, io
import numpy as np
import torch, torch.nn as nn
import onnx, onnxruntime as ort

torch.manual_seed(42)
print("torch       :", torch.__version__)
print("onnx        :", onnx.__version__)
print("onnxruntime :", ort.__version__)
import onnxsim; print("onnxsim     :", onnxsim.__version__)

---
# Part 1. Export 대상 모델 준비

### Step 1-1. 모델 정의 — 그리고 "왜 학습이 필요 없는가"

양자화 미니랩의 MiniCNN과 같은 구조를 사용합니다. 단, **학습은 하지 않습니다.**
오늘 다루는 것은 weight의 *값*이 아니라 그래프의 *구조*이기 때문입니다 —
Export·검증·단순화·Fallback 판정은 전부 구조의 문제이고, 랜덤 가중치로도 완벽히 실습됩니다.
(등가성 검증에서 "값"을 비교하지만, 그 값이 정확도일 필요는 없습니다.)

In [ ]:
class ConvBNReLU(nn.Sequential):
    def __init__(self, cin, cout, stride=1):
        super().__init__(nn.Conv2d(cin, cout, 3, stride, 1, bias=False),
                         nn.BatchNorm2d(cout), nn.ReLU())

class MiniCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.b1 = ConvBNReLU(3, 32)
        self.b2 = ConvBNReLU(32, 64, stride=2)
        self.b3 = ConvBNReLU(64, 128, stride=2)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(128, num_classes)
    def forward(self, x):
        return self.fc(self.pool(self.b3(self.b2(self.b1(x)))).flatten(1))

model = MiniCNN().eval()      # ★ Export도 eval() 필수 — BN/Dropout 동작 고정
dummy = torch.randn(1, 3, 32, 32)   # ★ 고정 shape 더미 입력: 이 shape이 그래프에 박제됨
print(model.b1)
print("\n더미 입력:", tuple(dummy.shape), "— batch=1 고정 (교안: dynamic_axes 금지)")

### Step 1-2. 그래프 해부 도구 상자 만들기

이후 실습 내내 쓸 3가지 도구를 먼저 만듭니다. ONNX 파일은 결국
**노드(연산) 리스트 + 초기값(weight) + 입출력 명세**를 담은 protobuf일 뿐입니다.

In [ ]:
def op_count(path):
    """연산자 종류별 개수 세기 — 그래프의 '지문'"""
    g = onnx.load(path).graph
    return collections.Counter(n.op_type for n in g.node)

def input_shape(path):
    """입력 텐서의 shape 읽기 — 고정 숫자인가, 심볼('batch' 등)인가"""
    inp = onnx.load(path).graph.input[0]
    return [d.dim_value if d.dim_value else d.dim_param
            for d in inp.type.tensor_type.shape.dim]

def show_nodes(path, max_n=15):
    """노드 나열 — 그래프를 위에서부터 읽기"""
    g = onnx.load(path).graph
    for i, n in enumerate(g.node[:max_n]):
        print(f"  [{i:>2}] {n.op_type:<22} {list(n.input)[:2]} → {list(n.output)}")
    if len(g.node) > max_n:
        print(f"  ... 외 {len(g.node)-max_n}개")

print("도구 3종 준비 완료: op_count / input_shape / show_nodes")

---
# Part 2. 정석 Export 레시피 — 그리고 등가성 검증 습관

### Step 2-1. [정석] `torch.onnx.export` 4대 인자

교안 3원칙(opset≥13 · 고정 shape · 표준 연산자)을 코드 인자로 옮기면 이렇습니다.

In [ ]:
torch.onnx.export(
    model, dummy, "mini_good.onnx",
    opset_version=13,             # ★ 원칙1: opset ≥ 13 (이유는 Part 3-2에서 증명)
    input_names=["input"],        # 텐서에 이름 부여 — 런타임·컴파일러가 참조
    output_names=["logits"],
    # dynamic_axes=... 를 '안 쓰는 것'이 원칙2 (기본값 = dummy shape 고정)
)
print("Export 완료 → mini_good.onnx")

### Step 2-2. 검증 1단계 — checker와 메타데이터

`onnx.checker`는 그래프가 ONNX 명세를 지키는지(타입·연결·shape 정합성) 검사합니다.
디퍼아이 컴파일러에 넣기 전 **가장 싼 1차 관문**입니다.

In [ ]:
m_onnx = onnx.load("mini_good.onnx")
onnx.checker.check_model(m_onnx)
print("✅ checker 통과")
print("opset      :", m_onnx.opset_import[0].version)
print("입력 shape :", input_shape("mini_good.onnx"), "← 전부 숫자 = 고정 shape ✅")
print("op 카운트  :", dict(op_count("mini_good.onnx")))

### Step 2-3. 🔍 관찰 — BatchNormalization이 사라졌다?!

위 op 카운트를 자세히 보세요. 모델에는 BN이 3개 있는데 그래프에는 **BatchNormalization 노드가 0개**입니다.

이유: **eval 모드로 export하면 torch가 BN을 Conv에 자동으로 접어 넣습니다(BN folding)** —
양자화 미니랩에서 `fuse_modules`로 손수 했던 일을 exporter가 해준 것입니다.
정말 그런지, 일부러 TRAINING 모드로 export해서 대조해 봅시다.

In [ ]:
# ⚠️ TRAINING 모드 export는 forward를 train 모드로 실행 → BN running 통계가 갱신되어 버림!
#    원본 model을 보호하기 위해 deepcopy 사본으로 export합니다 (실무에서도 필수 습관)
torch.onnx.export(copy.deepcopy(model), dummy, "mini_train_mode.onnx", opset_version=13,
                  input_names=["input"], output_names=["logits"],
                  training=torch.onnx.TrainingMode.TRAINING)

print("eval  모드 export:", dict(op_count("mini_good.onnx")))
print("train 모드 export:", dict(op_count("mini_train_mode.onnx")))
print("\n💡 train 모드에서는 BN이 그래프에 그대로 남습니다 (통계 갱신이 필요하니까).")
print("   → 'Export는 eval()에서'가 folding까지 챙기는 습관인 이유입니다.")
print("   → NPU 관점: 노드가 적을수록 컴파일 단순 + fusion 이득 (가상 NPU 실습 Part 10)")

### Step 2-4. 검증 2단계 — ★ 등가성 검증 (이 습관이 실무를 지킨다)

checker는 "문법"만 봅니다. **"수학이 같은가"**는 직접 확인해야 합니다:
같은 입력을 PyTorch와 onnxruntime에 넣어 출력을 대조합니다.
Day 2에서 `.tachyrt` 배포 후 "정확도 회귀 검증"을 하는 것과 같은 정신입니다.

In [ ]:
sess = ort.InferenceSession("mini_good.onnx", providers=["CPUExecutionProvider"])

x_test = torch.randn(1, 3, 32, 32)
with torch.no_grad():
    y_torch = model(x_test).numpy()
y_onnx = sess.run(None, {"input": x_test.numpy()})[0]

diff = np.abs(y_torch - y_onnx).max()
print(f"PyTorch vs ONNX 최대 오차: {diff:.2e}")
assert np.allclose(y_torch, y_onnx, atol=1e-4)
print("✅ 등가성 검증 통과 — Export가 수학적으로 안전합니다.")
print("\n⚠️ 이 검증을 생략하면 Part 3-3에서 볼 '조용히 틀리는 그래프'를 놓칩니다. 반드시 습관화!")

> **✅ Part 2 확인**
> - [ ] export 4대 인자(opset 13, 고정 dummy, 입출력 이름)를 쓸 수 있다
> - [ ] eval 모드 export가 BN folding까지 수행함을 대조 실험으로 확인했다
> - [ ] checker(문법) + ORT 대조(수학) 2단계 검증을 습관으로 이해했다

---
# Part 3. ★ 함정 실험실 — 교안의 6가지를 일부러 밟아보기

에러 메시지를 실전에서 처음 만나면 30분을 잃습니다. 여기서 미리 만나면 30초에 알아봅니다.

| # | 함정 | 이 실습에서 하는 일 |
| --- | --- | --- |
| 1 | Dynamic Shape | dynamic_axes로 export → 그래프에 박힌 심볼 확인 |
| 2 | Opset Version | ONNX 공식 스키마 DB를 조회해 "opset 13부터"를 **증명** |
| 3 | 추적(trace)의 함정 | 데이터 의존 분기 → **성공했는데 틀린 그래프** 재현 |
| 4 | Transformer 연산 | GELU·LayerNorm이 노드 폭탄으로 분해되는 현장 |
| 5 | ReduceMean 축 | 분해된 그래프에서 axes가 상수로 박제됐는지 확인 |
| 6 | Reshape 난무 | 지저분한 forward → 그래프 오염 확인 (Part 4에서 청소) |

### Step 3-1. [함정 1] Dynamic Shape — 그래프에 박히는 심볼

In [ ]:
torch.onnx.export(model, dummy, "mini_dynamic.onnx", opset_version=13,
                  input_names=["input"], output_names=["logits"],
                  dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}})  # ← 함정!

print("고정 shape :", input_shape("mini_good.onnx"))
print("동적 shape :", input_shape("mini_dynamic.onnx"))
print()
print("💡 첫 차원이 숫자 1이 아니라 문자열 'batch' — 크기 미정의 심볼입니다.")
print("   NPU 컴파일러는 이 그래프로 SRAM 타일 크기·사이클 스케줄을 '컴파일 시점에' 확정해야 하는데")
print("   (가상 NPU 실습 Part 5 타일링), 크기를 모르면 확정 불가 → 컴파일 실패.")
print("   교안: 'torch.onnx.export(dynamic_axes=None), 고정 shape 사용'")

### Step 3-2. [함정 2] Opset — "13부터"를 공식 스키마로 증명하기

교안: "*opset < 13이면 양자화 연산자 미지원*". 왜 하필 13일까요?
ONNX 라이브러리에는 모든 연산자의 버전 이력 DB(`onnx.defs`)가 내장되어 있습니다.
직접 조회해서 **per-channel 양자화의 핵심인 `axis` 속성이 언제 생겼는지** 확인합시다.

In [ ]:
for opset in [12, 13]:
    s = onnx.defs.get_schema("QuantizeLinear", max_inclusive_version=opset)
    print(f"opset {opset}에서의 QuantizeLinear: since_version={s.since_version}, "
          f"속성={list(s.attributes) or '없음'}")

print()
print("💡 opset ≤ 12: QuantizeLinear v10 — axis 속성 없음 → scale 1개(per-tensor)만 표현 가능")
print("   opset ≥ 13: QuantizeLinear v13 — axis 등장 → per-channel 양자화 표현 가능!")
print()
print("   양자화 미니랩 Part 4에서 'weight는 per-channel'이 표준임을 확인했었죠.")
print("   opset 12 이하로 export하면 그 per-channel 정보를 담을 그릇 자체가 없는 것입니다.")
print("   → 교안 'opset_version=13 이상 필수 명시'의 근거를 스키마로 확인했습니다 ✅")

### Step 3-3. [함정 3] ★ 가장 무서운 함정 — 성공했는데 틀린 그래프

`torch.onnx.export`는 기본적으로 **트레이싱(tracing)**: 더미 입력을 한 번 흘려보내며
"실행된 연산"만 기록합니다. 그래서 `if x.mean() > 0:` 같은 **데이터 의존 분기**를 만나면
에러 없이 — **더미가 지나간 한쪽 분기만** — 그래프에 박제합니다.

컴파일이 실패하면 30분을 잃지만, 이 함정은 **배포 후 특정 입력에서만 조용히 오답**을 냅니다.

In [ ]:
class BranchModel(nn.Module):
    """입력 평균의 부호에 따라 다른 연산을 하는 모델 — trace의 천적"""
    def __init__(self):
        super().__init__(); self.c = nn.Conv2d(3, 8, 3, padding=1)
    def forward(self, x):
        h = self.c(x)
        if x.mean() > 0:          # ← 데이터 의존 분기!
            return h * 2.0
        else:
            return h * -1.0

bm = BranchModel().eval()
dummy_pos = torch.ones(1, 3, 8, 8)              # mean > 0 인 더미로 export
torch.onnx.export(bm, dummy_pos, "branch.onnx", opset_version=13, input_names=["input"])
print("export '성공' — 경고(TracerWarning)만 뜨고 에러는 없습니다.")
print("그래프 ops:", dict(op_count("branch.onnx")), "← if도, -1 분기도 없음. Mul(x2)만 박제!")

In [ ]:
# 반대 분기로 가야 하는 입력으로 등가성 검증 → 참사 확인
sess_b = ort.InferenceSession("branch.onnx", providers=["CPUExecutionProvider"])
x_neg = -torch.ones(1, 3, 8, 8)                 # mean < 0

with torch.no_grad():
    y_pt = bm(x_neg).numpy()                     # PyTorch: -1x 분기 실행
y_ox = sess_b.run(None, {"input": x_neg.numpy()})[0]   # ONNX: 박제된 x2 실행

ratio = float(np.median(y_ox / (y_pt + 1e-12)))
print(f"원소별 비율의 중앙값  ONNX ÷ PyTorch = {ratio:+.2f}")
print(f"최대 오차: {np.abs(y_pt - y_ox).max():.3f}")
print()
print(f"해석: PyTorch는 (-1)x분기를 실행했고 ONNX는 박제된 (x2)를 실행 → 비율이 정확히 -2.00!")
print()
print("💥 checker도 통과, export도 성공 — 오직 '등가성 검증'만이 이 버그를 잡습니다 (Part 2-4 습관!)")
print("💡 해결책: 분기를 torch.where 등 텐서 연산으로 재작성, 또는 분기 로직을 후처리(CPU)로 이관.")
print("   교안 '커스텀 연산자 교체'도 같은 계열 — 참고로 C++ 커스텀 op는 여전히 symbolic 등록이 필요합니다.")

### Step 3-4. [함정 4] Transformer 연산 — GELU 하나가 노드 폭탄이 되는 현장

교안: "*LayerNorm · GELU는 NPU 일부 지원 — Fallback 발생 → Conv+BN+ReLU 구조로 교체*".
같은 역할의 블록 두 개를 export해서 그래프 크기를 비교합니다.

In [ ]:
class TransBlock(nn.Module):        # Transformer 스타일
    def __init__(self):
        super().__init__()
        self.c = nn.Conv2d(3, 16, 3, padding=1)
        self.ln = nn.LayerNorm([16, 32, 32])
        self.act = nn.GELU()
    def forward(self, x): return self.act(self.ln(self.c(x)))

class ConvBlock(nn.Module):         # CNN 스타일 (NPU 친화)
    def __init__(self):
        super().__init__()
        self.c = nn.Conv2d(3, 16, 3, padding=1)
        self.bn = nn.BatchNorm2d(16)
        self.act = nn.ReLU()
    def forward(self, x): return self.act(self.bn(self.c(x)))

torch.onnx.export(TransBlock().eval(), dummy, "block_trans.onnx", opset_version=13, input_names=["input"])
torch.onnx.export(ConvBlock().eval(),  dummy, "block_conv.onnx",  opset_version=13, input_names=["input"])

ct, cc = op_count("block_trans.onnx"), op_count("block_conv.onnx")
print(f"Conv+BN+ReLU 블록  : {sum(cc.values()):>2}개 노드 — {dict(cc)}")
print(f"Conv+LN+GELU 블록  : {sum(ct.values()):>2}개 노드 — {dict(ct)}")
print()
print("💡 LayerNorm → ReduceMean·Sub·Pow·Sqrt·Div... / GELU → Erf 조합으로 산산이 분해!")
print("   NPU가 Erf·Pow·Sqrt를 지원하지 않으면 이 노드들 전부가 Fallback 후보가 됩니다.")
print("   (Part 5에서 스캐너로 정확히 몇 %인지 계산합니다)")

### Step 3-5. [함정 5] ReduceMean의 축(axes) — 상수인가, 동적인가

교안: "*축 파라미터 정적 필수 — 동적 지정 시 실패*".
방금 만든 Transformer 블록 그래프에서 ReduceMean 노드를 찾아,
axes가 **상수로 박제**되어 있는지 직접 확인합니다. (PyTorch에서 `dim=`을 파이썬 상수로 쓰면
trace가 상수로 박제해 줍니다 — 텐서 값으로 축을 정하는 코드가 위험한 경우입니다.)

In [ ]:
g = onnx.load("block_trans.onnx").graph
for n in g.node:
    if n.op_type == "ReduceMean":
        axes = None
        for a in n.attribute:
            if a.name == "axes":
                axes = list(a.ints)
        print(f"ReduceMean 노드 '{n.name or n.output[0]}' → axes = {axes} (그래프에 박힌 상수)")
print()
print("✅ axes가 attribute 상수로 고정 — 컴파일러가 읽을 수 있는 형태입니다.")
print("⚠️ 만약 축이 입력 텐서 값에 따라 달라지는 코드였다면 이 자리가 동적이 되어 컴파일 실패.")

### Step 3-6. [함정 6] Reshape 난무 — 지저분한 forward가 만드는 그래프 오염

view/permute를 습관적으로 왕복하는 forward를 export하면 그래프에 무의미한
Reshape·Constant가 쌓입니다. 일부러 지저분한 모델을 만들어 봅시다. (청소는 Part 4에서!)

In [ ]:
class MessyModel(nn.Module):
    """수학적으로는 Conv→Flatten→FC와 동일하지만 forward가 지저분한 모델"""
    def __init__(self):
        super().__init__()
        self.c = nn.Conv2d(3, 16, 3, padding=1)
        self.fc = nn.Linear(16 * 32 * 32, 10)
    def forward(self, x):
        h = self.c(x)
        b, c, hh, ww = h.shape
        h = h.view(b, c, hh * ww).permute(0, 2, 1)      # 갔다가...
        h = h.permute(0, 2, 1).reshape(b, c, hh, ww)    # ...그대로 돌아옴 (무의미!)
        h = h.flatten(1)
        h = h.view(b, h.shape[1])                        # 이미 그 shape인데 또 reshape
        return self.fc(h)

torch.onnx.export(MessyModel().eval(), dummy, "messy.onnx", opset_version=13, input_names=["input"])
cm = op_count("messy.onnx")
print(f"messy.onnx: 총 {sum(cm.values())}개 노드 — {dict(cm)}")
print()
show_nodes("messy.onnx")
print("\n💡 수학적으로는 Conv/Flatten/Gemm 3개면 충분한데 Reshape·Constant가 그래프를 오염시켰습니다.")
print("   교안: 'Export된 그래프에 불필요한 reshape이 수십개 → onnx-simplifier로 단순화 필수'")

> **✅ Part 3 확인**
> - [ ] 동적 shape 심볼이 그래프에 어떻게 박히는지, 왜 NPU가 거부하는지 설명할 수 있다
> - [ ] "opset 13 = per-channel axis의 탄생"을 스키마 조회로 증명했다
> - [ ] "성공했지만 틀린 그래프"를 재현했고, 등가성 검증이 유일한 방어선임을 체감했다
> - [ ] GELU/LayerNorm 분해와 Reshape 오염을 눈으로 확인했다

---
# Part 4. onnx-simplifier 실전 — 그래프 청소

### Step 4-1. simplify 적용 — 상수 접기 + 무의미 연산 제거

In [ ]:
from onnxsim import simplify

simplified, check_ok = simplify(onnx.load("messy.onnx"))
onnx.save(simplified, "messy_simp.onnx")

before, after = op_count("messy.onnx"), op_count("messy_simp.onnx")
print(f"simplify 전: {sum(before.values())}개 노드 — {dict(before)}")
print(f"simplify 후: {sum(after.values()):>2}개 노드 — {dict(after)}")
print(f"내부 등가성 체크: {check_ok}")
print()
show_nodes("messy_simp.onnx")
print("\n💡 무의미한 Reshape 왕복이 전부 제거되고 수학적 최소형(Conv→Flatten→Gemm)만 남았습니다.")

### Step 4-2. 청소 후에도 등가성 검증 — "빨래가 옷을 줄이지 않았는지"

simplifier도 변환기입니다. 변환기를 거쳤으면 **무조건** 등가성 검증 (Part 2-4 습관 반복).

In [ ]:
s1 = ort.InferenceSession("messy.onnx",      providers=["CPUExecutionProvider"])
s2 = ort.InferenceSession("messy_simp.onnx", providers=["CPUExecutionProvider"])

xn = torch.randn(1, 3, 32, 32).numpy()
n1, n2 = s1.get_inputs()[0].name, s2.get_inputs()[0].name
d = np.abs(s1.run(None, {n1: xn})[0] - s2.run(None, {n2: xn})[0]).max()
print(f"simplify 전후 최대 출력 차이: {d:.2e}")
assert d < 1e-5
print("✅ 노드 수는 1/3, 수학은 그대로 — 안전한 청소였습니다.")
print("\n✏️ 직접 해보기: Part 2의 mini_good.onnx에도 simplify를 적용해 보세요.")
print("   이미 깨끗한 그래프라면 변화가 거의 없을 것입니다 — 그것도 확인해 볼 가치가 있습니다.")

---
# Part 5. ★ Fallback 스캐너 직접 만들기 — 컴파일 전에 매핑률 예측

Day 2 체크포인트 ③은 "*NPU 활용률(연산자 매핑률) 90% 이상*"이었습니다.
컴파일러를 돌리기 전에 ONNX 그래프만 보고 매핑률을 예측하는 도구를 직접 만듭니다.
가상 NPU 실습 Part 8(Fallback 비용)의 "어떤 노드가 CPU로 떨어질까?"에 대한 답이기도 합니다.

### Step 5-1. BlackSwan 지원 연산자 화이트리스트 → 스캐너

교안 스펙시트의 지원 연산자(Conv1D/2D, TransposeConv, DW-Conv, Pooling, FC, BN, 활성함수)를
ONNX op 이름으로 옮겨 화이트리스트를 만들고, 그래프를 순회하며 판정합니다.

In [ ]:
# 교안 BlackSwan 지원 연산자를 ONNX op_type으로 매핑한 화이트리스트 (교육용 근사)
NPU_SUPPORTED = {
    "Conv", "ConvTranspose",                       # Conv1D/2D, TransposeConv, DW-Conv(Conv의 group)
    "MaxPool", "AveragePool", "GlobalAveragePool", # Pooling
    "Gemm", "MatMul",                              # FC
    "BatchNormalization",                          # BN
    "Relu", "LeakyRelu", "Sigmoid", "Tanh", "Clip",# 활성함수
    "Add", "Mul", "Concat", "Flatten", "Reshape",  # 기본 텐서 조작 (대부분 NPU가 흡수)
}

def scan_fallback(path, title=""):
    cnt = op_count(path)
    total = sum(cnt.values())
    fallback = {op: n for op, n in cnt.items() if op not in NPU_SUPPORTED}
    n_fb = sum(fallback.values())
    util = (total - n_fb) / total * 100
    print(f"═══ Fallback 스캔: {title or path} ═══")
    print(f"총 노드 {total}개 | NPU 매핑 {total-n_fb}개 | Fallback {n_fb}개")
    if fallback:
        for op, n in sorted(fallback.items(), key=lambda kv: -kv[1]):
            print(f"  ⚠️ {op:<14} x{n}  → CPU Fallback 후보")
    print(f"예상 NPU 매핑률: {util:.0f}%  {'✅ 목표(90%) 달성' if util >= 90 else '❌ 목표 미달 — 구조 수정 필요'}")
    print()
    return util

u_conv  = scan_fallback("block_conv.onnx",  "Conv+BN+ReLU 블록")
u_trans = scan_fallback("block_trans.onnx", "Conv+LN+GELU 블록")

### Step 5-2. 처방 적용 — GELU→ReLU 교체 후 재스캔

교안 Fallback 대응 ①"연산자 교체"를 적용하고 스캐너로 개선을 확인합니다.
(가상 NPU 실습 Part 8에서 지연 시간으로 봤던 것을, 이번엔 그래프 수준에서 봅니다.)

In [ ]:
class FixedBlock(nn.Module):
    """TransBlock의 처방판: LayerNorm→BatchNorm, GELU→ReLU"""
    def __init__(self):
        super().__init__()
        self.c = nn.Conv2d(3, 16, 3, padding=1)
        self.bn = nn.BatchNorm2d(16)     # LN → BN
        self.act = nn.ReLU()             # GELU → ReLU
    def forward(self, x): return self.act(self.bn(self.c(x)))

torch.onnx.export(FixedBlock().eval(), dummy, "block_fixed.onnx",
                  opset_version=13, input_names=["input"])
u_fixed = scan_fallback("block_fixed.onnx", "처방 적용 블록 (BN+ReLU 교체)")

print(f"매핑률 변화: {u_trans:.0f}% → {u_fixed:.0f}%")
print("💡 물론 공짜는 아닙니다 — 연산자 교체 후에는 재학습으로 정확도를 회복해야 합니다.")
print("   'NPU를 고려한 모델 커스터마이징'(교안 Day 2 Best Practice)이 바로 이 작업입니다.")
print("\n✏️ 직접 해보기: mini_good.onnx와 messy_simp.onnx도 스캔해 보세요. 몇 %가 나오나요?")

> **✅ Part 5 확인**
> - [ ] 지원 연산자 화이트리스트로 그래프의 매핑률을 예측하는 도구를 만들었다
> - [ ] 연산자 교체 처방이 매핑률에 미치는 효과를 정량 확인했다
> - [ ] 실제 컴파일러 로그의 "연산자 매핑률(Fallback) 확인"(Day 2 ⑤단계)이 무엇을 세는지 이해했다

---
# Part 6. Netron 시각화 — 그래프를 눈으로 확인

지금까지는 코드로 해부했지만, 실무에서는 **Netron**(교안 컴파일 ③단계 도구)으로
그래프를 눈으로 봅니다. 파일을 내려받아 웹에서 열어 봅시다.

### Step 6-1. 파일 다운로드

In [ ]:
try:
    from google.colab import files
    for f in ["mini_good.onnx", "block_trans.onnx", "messy.onnx", "messy_simp.onnx"]:
        files.download(f)
    print("4개 파일 다운로드 시작 — 브라우저 저장 대화상자를 확인하세요.")
except ImportError:
    print("(Colab 밖 환경 — 파일은 현재 폴더에 있습니다: mini_good/block_trans/messy/messy_simp.onnx)")

### Step 6-2. https://netron.app 에서 열고 — 관찰 체크리스트

브라우저에서 **netron.app** 접속 → "Open Model" → 내려받은 파일 선택. (설치 불필요, 로컬에서 렌더링)

| 파일 | 관찰 포인트 |
| --- | --- |
| `mini_good.onnx` | ① input 클릭 → shape이 [1,3,32,32] 고정인지 ② Conv 클릭 → W의 shape과 BN folding 흔적 ③ 전체 구조가 일직선인지 |
| `block_trans.onnx` | GELU가 Erf·Mul·Add 사슬로, LayerNorm이 ReduceMean·Sub·Pow·Sqrt·Div로 분해된 모습 — Part 3-4에서 숫자로 본 것을 그림으로 |
| `messy.onnx` vs `messy_simp.onnx` | 두 창을 나란히 — Reshape 덩어리가 사라진 before/after |

> 💡 Day 2 본 실습에서 컴파일이 실패하면, 가장 먼저 할 일이 **Netron으로 ONNX를 열어
> 미지원 노드·동적 shape을 눈으로 찾는 것**입니다. 오늘의 관찰 습관이 그대로 무기가 됩니다.

---
# Part 7. 리포트 과제 & Day 2 연결

## 제출 과제 — `onnx_report.md`

**A. 결과표 채우기**

| 파일 | 총 노드 | 주요 op | 예상 매핑률 |
| --- | --- | --- | --- |
| mini_good.onnx | | | |
| block_conv.onnx | | | 100% |
| block_trans.onnx | | | |
| block_fixed.onnx | | | |
| messy.onnx → messy_simp.onnx | → | | |

**B. 분석 질문 (각 2~3문장)**
1. eval 모드 export에서 BN 노드가 사라진 이유와, 그것이 NPU에 이득인 이유는?
2. 함정 3(분기 박제)은 checker·컴파일러 모두 통과할 수 있다. 이 버그를 잡을 수 있는 유일한 검증은 무엇이며, 파이프라인 어느 지점마다 넣어야 하는가?
3. opset 13의 `axis` 속성과 양자화 미니랩의 per-channel scale은 어떤 관계인가?
4. YOLOv9의 NMS는 왜 스캐너 화이트리스트에 넣지 않는가? (힌트: 교안 "NMS는 CPU 실행(NPU 밖)")

**C. 스크린샷**: Netron에서 연 block_trans.onnx의 GELU 분해 부분, messy 전후 비교, 스캐너 출력

## 오늘 배운 것 ↔ Day 2 본 실습 대응표

| 이 실습 | Day 2 본 실습 (디퍼아이) |
| --- | --- |
| `torch.onnx.export` 정석 인자 | `compile_linux.py`의 ONNX 변환 단계 + `ONNX_INPUT_SHAPE` 고정 |
| checker + ORT 등가성 검증 | 배포 후 "정확도 회귀 검증" (컴파일 ⑥단계) |
| Fallback 스캐너 (예측) | 컴파일러 로그의 실제 연산자 매핑률 (⑤단계) |
| onnx-simplifier | 컴파일러 그래프 최적화 + "ONNX surgeon" 개념 (Best Practice ③) |
| Netron 관찰 | 컴파일 실패 시 1차 디버깅 도구 |

## ✏️ 심화 도전 과제 (선택)

1. **양자화 미니랩 연결**: 미니랩에서 학습한 FP32 MiniCNN을 이 레시피로 export하고 스캐너·simplify·Netron까지 전 과정 적용 (진짜 weight로 등가성 검증)
2. **스캐너 고도화**: Fallback 노드가 그래프의 앞/중간/끝 어디에 몰려 있는지 위치 리포트 추가 — 가상 NPU 실습 Part 8 "전환 횟수"와 연결해 예상 NPU↔CPU 왕복 횟수까지 추정
3. **함정 5 심화**: `torch.mean(x, dim=some_tensor_value)` 형태로 축이 동적이 되는 코드를 만들어 export가 어떻게 되는지 실험
4. **Depthwise 확인**: `groups=cin`인 Conv를 export해 그래프에서 group 속성을 찾고, 스캐너가 DW-Conv를 Conv로 올바르게 인식하는지 확인

---
수고하셨습니다! 🎉 이제 **학습 → 양자화 → Export → 그래프 검증**까지,
`.tachyrt` 컴파일 직전의 전 구간을 여러분 손으로 통과했습니다.
Day 2 본 실습에서 디퍼아이 컴파일러가 뱉는 로그 한 줄 한 줄이 이제 읽히기 시작할 것입니다.
